# Common Crawl News Explorer

Reference: see `API_USAGE.md` in this folder.

In [1]:
from datetime import datetime, timezone

# Human-readable source label used in notebook output.
SOURCE_NAME = "Common Crawl News (CC-NEWS)"
# Data host used for both manifest files and WARC objects.
DATA_ROOT = "https://data.commoncrawl.org"
# Use current UTC year/month by default so the notebook stays current over time.
CURRENT_UTC = datetime.now(timezone.utc)
DEFAULT_YEAR = f"{CURRENT_UTC.year:04d}"
DEFAULT_MONTH = f"{CURRENT_UTC.month:02d}"
DEFAULT_MANIFEST_URL = (
    f"{DATA_ROOT}/crawl-data/CC-NEWS/{DEFAULT_YEAR}/{DEFAULT_MONTH}/warc.paths.gz"
)
CAPABILITIES = [
    "Massive web-scale corpus with historical news snapshots.",
    "Monthly CC-NEWS WARC manifest files for bulk extraction.",
    "Open dataset suitable for custom archive mining pipelines.",
    "Raw WARC records can be parsed into article-level datasets.",
]

print(SOURCE_NAME)
print("Preview only: this cell does not fetch records.")
print("Capabilities:")
for item in CAPABILITIES:
    print("-", item)
print("\nManifest URL preview (current UTC month):")
print(DEFAULT_MANIFEST_URL)
print("\nImportant: warc.paths.gz returns file paths, not article JSON rows.")
print(
    "To get article records, download one listed .warc.gz file and parse WARC entries."
)

Common Crawl News (CC-NEWS)
Preview only: this cell does not fetch records.
Capabilities:
- Massive web-scale corpus with historical news snapshots.
- Monthly CC-NEWS WARC manifest files for bulk extraction.
- Open dataset suitable for custom archive mining pipelines.
- Raw WARC records can be parsed into article-level datasets.

Manifest URL preview (current UTC month):
https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/03/warc.paths.gz

Important: warc.paths.gz returns file paths, not article JSON rows.
To get article records, download one listed .warc.gz file and parse WARC entries.


In [2]:
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import gzip
import json
import time

# Use default from preview cell unless user overrides MANIFEST_URL manually.
manifest_url = globals().get(
    "MANIFEST_URL",
    globals().get(
        "DEFAULT_MANIFEST_URL",
        "https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/03/warc.paths.gz",
    ),
)
print("Live request URL:")
print(manifest_url)

compressed_payload = None
last_error = None
for attempt_index, delay_seconds in enumerate([0, 3, 6], start=1):
    if delay_seconds > 0:
        print(f"Waiting {delay_seconds}s before retry...")
        time.sleep(delay_seconds)
    try:
        request = Request(manifest_url, headers={"User-Agent": "news-api-explorer/1.0"})
        with urlopen(request, timeout=30) as response:
            compressed_payload = response.read()
        print(f"Fetch succeeded on attempt {attempt_index}.")
        break
    except HTTPError as error:
        last_error = error
        print(f"Attempt {attempt_index} failed with HTTP {error.code}.")
        if error.code != 429:
            break
    except Exception as error:
        last_error = error
        print(f"Attempt {attempt_index} failed: {error}")
        break

if compressed_payload is None:
    print("No live payload returned.")
    print(f"Last error: {last_error}")
else:
    decoded_text = gzip.decompress(compressed_payload).decode("utf-8")
    paths = [line.strip() for line in decoded_text.splitlines() if line.strip()]
    print(f"\nNews WARC paths returned: {len(paths)}")
    print("This list is expected: it is a monthly manifest of WARC files.")

    sample_paths = paths[:5]
    print("\nSample paths:")
    for index, warc_path in enumerate(sample_paths, start=1):
        print(f"{index}. {warc_path}")

    first_warc_url = (
        f"https://data.commoncrawl.org/{sample_paths[0]}" if sample_paths else None
    )
    if first_warc_url:
        print("\nFirst full WARC URL example:")
        print(first_warc_url)

    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "commoncrawl_live_paths_sample.json"
    payload = {
        "source_url": manifest_url,
        "total_paths": len(paths),
        "sample_paths": sample_paths,
        "first_warc_url": first_warc_url,
        "note": "warc.paths.gz is a manifest of WARC object paths, not an article JSON API response.",
    }
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"\nSaved payload to: {output_path.resolve()}")

Live request URL:
https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/03/warc.paths.gz
Fetch succeeded on attempt 1.

News WARC paths returned: 53
This list is expected: it is a monthly manifest of WARC files.

Sample paths:
1. crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301003313-06997.warc.gz
2. crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301041355-06998.warc.gz
3. crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301071259-06999.warc.gz
4. crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301093057-07000.warc.gz
5. crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301114033-07001.warc.gz

First full WARC URL example:
https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/03/CC-NEWS-20260301003313-06997.warc.gz

Saved payload to: /Users/gwh/projects/news/notebooks/api_explorer/commoncrawl/outputs/commoncrawl_live_paths_sample.json


In [3]:
from pathlib import Path
import json

# Build cc-pyspark-compatible manifest files from the live manifest output.
# cc-pyspark expects one file path per line in the input manifest.
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Prefer in-memory `paths` from the live fetch cell; fallback to saved sample JSON.
if "paths" in globals() and isinstance(paths, list) and paths:
    manifest_paths = paths
else:
    sample_json_path = output_dir / "commoncrawl_live_paths_sample.json"
    if sample_json_path.exists():
        sample_payload = json.loads(sample_json_path.read_text(encoding="utf-8"))
        manifest_paths = sample_payload.get("sample_paths", [])
    else:
        manifest_paths = []

if not manifest_paths:
    print("No manifest paths available. Run the live manifest cell first.")
else:
    # Write relative paths for cc-pyspark + --input_base_url workflow.
    relative_manifest_path = output_dir / "cc_news_manifest_relative.txt"
    relative_manifest_path.write_text(
        "\n".join(manifest_paths) + "\n", encoding="utf-8"
    )

    # Write full HTTPS URLs as an alternative input style.
    full_urls = [f"https://data.commoncrawl.org/{path}" for path in manifest_paths]
    https_manifest_path = output_dir / "cc_news_manifest_https.txt"
    https_manifest_path.write_text("\n".join(full_urls) + "\n", encoding="utf-8")

    print("Generated cc-pyspark manifest files:")
    print(f"- Relative paths: {relative_manifest_path.resolve()}")
    print(f"- Full HTTPS URLs: {https_manifest_path.resolve()}")

    print("\ncc-pyspark run pattern (from commoncrawl/cc-pyspark):")
    print("1) Clone cc-pyspark and enter it.")
    print("2) Install Spark + cc-pyspark dependencies.")
    print("3) Run a Spark job with your manifest.")

    print("\nExample command (relative manifest + --input_base_url):")
    print(
        "spark-submit ./server_count.py "
        "--num_output_partitions 1 --log_level WARN "
        f"--input_base_url https://data.commoncrawl.org/ {relative_manifest_path.resolve()} cc_news_servernames"
    )

    print("\nExample command (convert WARC -> WET text index):")
    print(
        "spark-submit ./wet_extractor.py "
        "--num_output_partitions 1 --log_level WARN "
        f"--input_base_url https://data.commoncrawl.org/ "
        f"--output_base_url file://{(output_dir / 'wet_output').resolve()} "
        f"{relative_manifest_path.resolve()} cc_news_wet_index"
    )

    print(
        "\nTip: run `spark-submit <job>.py --help` inside cc-pyspark to verify arguments for your version."
    )

Generated cc-pyspark manifest files:
- Relative paths: /Users/gwh/projects/news/notebooks/api_explorer/commoncrawl/outputs/cc_news_manifest_relative.txt
- Full HTTPS URLs: /Users/gwh/projects/news/notebooks/api_explorer/commoncrawl/outputs/cc_news_manifest_https.txt

cc-pyspark run pattern (from commoncrawl/cc-pyspark):
1) Clone cc-pyspark and enter it.
2) Install Spark + cc-pyspark dependencies.
3) Run a Spark job with your manifest.

Example command (relative manifest + --input_base_url):
spark-submit ./server_count.py --num_output_partitions 1 --log_level WARN --input_base_url https://data.commoncrawl.org/ /Users/gwh/projects/news/notebooks/api_explorer/commoncrawl/outputs/cc_news_manifest_relative.txt cc_news_servernames

Example command (convert WARC -> WET text index):
spark-submit ./wet_extractor.py --num_output_partitions 1 --log_level WARN --input_base_url https://data.commoncrawl.org/ --output_base_url file:///Users/gwh/projects/news/notebooks/api_explorer/commoncrawl/outputs